In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Sintetizar operações unitárias

<details>
<summary><b>Versões dos pacotes</b></summary>

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou mais recentes.

```
qiskit[all]~=2.3.0
```
</details>
Uma operação unitária descreve uma mudança que preserva a norma de um sistema quântico.
Para $n$ qubits, essa mudança é descrita por uma matriz complexa de dimensão $2^n \times 2^n$, $U$, cujo adjunto é igual ao inverso, ou seja, $U^\dagger U = \mathbb{1}$.

Sintetizar operações unitárias específicas em um conjunto de Gates quânticos é uma tarefa fundamental usada, por exemplo, no design e na aplicação de algoritmos quânticos ou na compilação de Circuits quânticos.

Embora a síntese eficiente seja possível para certas classes de unitários — como aqueles compostos por Gates de Clifford ou com estrutura de produto tensorial — a maioria dos unitários não se enquadra nessas categorias.
Para matrizes unitárias gerais, a síntese é uma tarefa complexa com custos computacionais que crescem exponencialmente com o número de qubits.
Portanto, se você conhece uma decomposição eficiente para o unitário que deseja implementar, ela provavelmente será melhor do que uma síntese geral.

> **Note:** Se nenhuma decomposição estiver disponível, o Qiskit SDK fornece as ferramentas para encontrar uma.
>     No entanto, observe que isso geralmente gera Circuits profundos que podem ser inadequados para execução em computadores quânticos ruidosos.

In [1]:
import numpy as np
from qiskit import QuantumCircuit

U = 0.5 * np.array(
    [[1, 1, 1, 1], [-1, 1, -1, 1], [-1, -1, 1, 1], [-1, 1, 1, -1]]
)

circuit = QuantumCircuit(2)
circuit.unitary(U, circuit.qubits)

## Re-synthesis for circuit optimization

Sometimes it is beneficial to re-synthesize a long series of single- and two-qubit gates, if the length can be reduced. For example, the following circuit uses three two-qubit gates.

In [2]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit

qreg_q = QuantumRegister(2, "q")
creg_c = ClassicalRegister(4, "c")
circuit = QuantumCircuit(qreg_q, creg_c)

circuit.h(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.sx(qreg_q[1])
circuit.cz(qreg_q[0], qreg_q[1])
circuit.x(qreg_q[1])
circuit.x(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.h(qreg_q[0])
circuit.draw("mpl")

<Image src="../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg" alt="Output of the previous code cell" />

## Re-síntese para otimização de Circuit
Às vezes é vantajoso re-sintetizar uma longa sequência de Gates de um e dois qubits, caso o comprimento possa ser reduzido. Por exemplo, o Circuit a seguir usa três Gates de dois qubits.

In [3]:
from qiskit.quantum_info import Operator

# compute unitary matrix of circuit
U = Operator(circuit)

# re-synthesize
better_circuit = QuantumCircuit(2)
better_circuit.unitary(U, range(2))
better_circuit.decompose().draw()

global phase: 6.2071
      ┌───────────────┐         ┌────────────────┐ 
q_0: ─┤ U(π/2,π/2,-π) ├────■────┤ U(π/2,-π,-π/2) ├─
     ┌┴───────────────┴─┐┌─┴─┐┌─┴────────────────┴┐
q_1: ┤ U(1.7229,π/2,-π) ├┤ X ├┤ U(π/2,0.15207,-π) ├
     └──────────────────┘└───┘└───────────────────┘

![Saída da célula de código anterior](../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg)

No entanto, após a re-síntese com o código a seguir, ele precisa apenas de um único Gate CX. (Aqui usamos o método `QuantumCircuit.decompose()` para visualizar melhor os Gates usados para re-sintetizar o unitário.)